In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class ReviewState(TypedDict):
    topic: str
    draft: str
    human_feedback: str
    final_answer: str

In [5]:
def write_draft(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You write short, clear explanations."),
        HumanMessage(content=f"Write a short answer about: {state['topic']}"),
    ])
    return {"draft": response.content}


def get_human_feedback(state: ReviewState) -> dict:
    feedback = interrupt(
        {
            "draft": state["draft"],
            "question": "Review this draft. What should be changed before finalizing it?",
        }
    )
    return {"human_feedback": feedback}


def revise_with_feedback(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You revise drafts using human feedback."),
        HumanMessage(
            content=(
                f"Draft:\n{state['draft']}\n\n"
                f"Human feedback:\n{state['human_feedback']}\n\n"
                "Return the final improved answer."
            )
        ),
    ])
    return {"final_answer": response.content}


builder = StateGraph(ReviewState)
builder.add_node("write_draft", write_draft)
builder.add_node("get_human_feedback", get_human_feedback)
builder.add_node("revise_with_feedback", revise_with_feedback)

builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "get_human_feedback")
builder.add_edge("get_human_feedback", "revise_with_feedback")
builder.add_edge("revise_with_feedback", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Human-in-the-loop graph compiled successfully.")

topic = input("Enter a topic for the draft: ")
config = {"configurable": {"thread_id": "hitl-demo"}}
initial_state = {
    "topic": topic,
    "draft": "",
    "human_feedback": "",
    "final_answer": "",
}

paused_result = graph.invoke(initial_state, config=config)
interrupt_value = paused_result["__interrupt__"][0].value

print("\nDraft for human review:")
print(interrupt_value["draft"])
print("\nQuestion:")
print(interrupt_value["question"])

human_feedback = input("Enter your feedback for the draft: ")

final_result = graph.invoke(
    Command(resume=human_feedback),
    config=config,
)

print("\nFinal answer after human feedback:")
print(final_result["final_answer"])

Human-in-the-loop graph compiled successfully.

Draft for human review:
Education is the process of acquiring knowledge, skills, values, and attitudes through teaching, learning, and experience. It encompasses formal systems (schools, universities) and informal methods (self-study, mentorship), aiming to foster critical thinking, personal growth, and societal progress. Education empowers individuals to navigate the world, contribute to their communities, and adapt to changing environments. Access to quality education is a cornerstone of equity and global development.

Question:
Review this draft. What should be changed before finalizing it?

Final answer after human feedback:
**Revised:**  
Education is the way people gain knowledge, skills, values, and attitudes through teaching, learning, and experience. It includes formal systems like schools and universities, as well as informal methods like self-study and mentorship. It helps develop critical thinking, personal growth, and progres